In [ ]:
!pip install librosa soundfile numpy

In [ ]:
from pathlib import Path

DATASET_DIR = Path("/content/drive/MyDrive/실전프로젝트/데이터 분석/data/New_Sample (oldman_voice)")

print(DATASET_DIR.exists())
print(DATASET_DIR)

In [ ]:
label_files = list(DATASET_DIR.rglob("*.json"))
audio_files = list(DATASET_DIR.rglob("*.wav"))

print("라벨 JSON 개수:", len(label_files))
print("음성 WAV 개수:", len(audio_files))

In [ ]:
import unicodedata

audio_map = {
    unicodedata.normalize("NFC", p.name): p
    for p in audio_files
}

In [ ]:
import json
import random

label_path = random.choice(label_files)

with open(label_path, "r", encoding="utf-8") as f:
    data = json.load(f)

utter_info = data.get("발화정보", {})
conv_info = data.get("대화정보", {})
speaker_info = data.get("녹음자정보", {})

file_name = utter_info.get("fileNm")
file_name = unicodedata.normalize("NFC", file_name)

audio_path = audio_map.get(file_name)

sample_item = {
    "label_path": str(label_path),
    "audio_path": str(audio_path) if audio_path is not None else None,
    "file_name": file_name,
    "reference_text": utter_info.get("stt"),
    "record_time": utter_info.get("recrdTime"),
    "quality": utter_info.get("recrdQuality"),
    "region": conv_info.get("cityCode"),
    "topic": conv_info.get("convrsThema", "").strip(),
    "gender": speaker_info.get("gender"),
    "age": speaker_info.get("age"),
}

print("음성 파일 매칭 여부:", audio_path is not None)
print("파일명:", sample_item["file_name"])
print("지역:", sample_item["region"])
print("나이/성별:", sample_item["age"], sample_item["gender"])
print("주제:", sample_item["topic"])
print("정답 전사:", sample_item["reference_text"])
print("음성 경로:", sample_item["audio_path"])

In [ ]:
from IPython.display import Audio, display

display(Audio(sample_item["audio_path"]))
print("정답 전사:", sample_item["reference_text"])

In [ ]:
audio_path = sample_item["audio_path"]

print("선택된 wav 파일 경로:")
print(audio_path)

print("\n정답 전사:")
print(sample_item["reference_text"])

In [ ]:
import librosa
import numpy as np

def detect_first_sound_time(
    audio_path,
    sample_rate=16000,
    frame_duration=0.05,
    threshold_ratio=0.10,
    min_sound_duration=0.15
):
    """
    wav 파일 앞부분에서 처음 소리가 감지되는 시점을 초 단위로 반환한다.

    audio_path: 음성 파일 경로
    sample_rate: 음성 샘플레이트
    frame_duration: 소리 크기를 확인할 구간 길이(초)
    threshold_ratio: 전체 최대 소리 크기 대비 기준 비율
    min_sound_duration: 소리가 이 시간 이상 이어져야 실제 소리로 인정
    """

    y, sr = librosa.load(audio_path, sr=sample_rate, mono=True)

    if len(y) == 0:
        return None

    frame_length = int(sr * frame_duration)
    min_sound_frames = max(1, int(min_sound_duration / frame_duration))

    max_amplitude = np.max(np.abs(y))

    if max_amplitude == 0:
        return None

    threshold = max_amplitude * threshold_ratio

    sound_flags = []

    for start in range(0, len(y), frame_length):
        frame = y[start:start + frame_length]

        if len(frame) == 0:
            continue

        frame_amplitude = np.mean(np.abs(frame))
        is_sound = frame_amplitude >= threshold
        sound_flags.append(is_sound)

    consecutive = 0

    for i, is_sound in enumerate(sound_flags):
        if is_sound:
            consecutive += 1

            if consecutive >= min_sound_frames:
                first_frame_index = i - min_sound_frames + 1
                first_sound_time = first_frame_index * frame_duration
                return round(first_sound_time, 2)
        else:
            consecutive = 0

    return None

In [ ]:
first_sound_time = detect_first_sound_time(audio_path)

print("파일명:", sample_item["file_name"])
print("지역:", sample_item["region"])
print("정답 전사:", sample_item["reference_text"])
print("첫 소리 감지 시점:", first_sound_time, "초")

In [ ]:
display(Audio(audio_path))
print("첫 소리 감지 시점:", first_sound_time, "초")

In [ ]:
def calculate_response_time_from_audio(audio_path):
    return detect_first_sound_time(audio_path)